# Chapter 2: Agent Control Flow

Estimated time: about 8 hours.

Prerequisites: Chapter 1 (the ReAct loop, mock tools, and `agentlib.llm_client` built
there are all reused here).

Interview category this chapter maps to: multi-agent architecture questions, such as when
multiple agents help vs. hurt, what a subagent actually is, and diagnosing which failure
mode ("it's slow," "it's stuck," "it's wrong") you're looking at from a symptom description
alone.

## Setup

In [ ]:
import sys
from pathlib import Path

_repo_root = Path.cwd()
if not (_repo_root / "agentlib").is_dir():
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

import difflib
import json
import random

from agentlib import llm_client
from agentlib.grading import check
from agentlib.tracing import Tracer

random.seed(42)
print(f"Repo root on sys.path: {_repo_root}")
print(f"LLM_PROVIDER = {llm_client.LLM_PROVIDER!r}, HAS_KEY = {llm_client.HAS_KEY}")


## Section 1: Definitions

### Subagent context isolation

A subagent receives a bounded task in a fresh context, executes independently, and returns
a compressed synthesis to the orchestrator. The orchestrator never sees the subagent's
intermediate reasoning, tool calls, or observations. Two properties define isolation:
fresh context *in* (nothing from the parent leaks down) and compressed result *out*
(nothing from the child leaks up except the answer).

**Real-world examples:**

- **GitHub Copilot Workspace** (GitHub, 2024) decomposes a coding task into planning,
  implementation, and validation. Each phase runs as a separate subagent with its own
  context. The planner never sees raw compiler output; the validator never sees draft code
  that was discarded.
- **Devin** (Cognition, 2024) maintains internal subagents for shell commands, browser
  navigation, and code editing. Each subagent operates in isolation so a browser-navigation
  failure does not corrupt the editor's state.
- **Customer support escalation systems** route a ticket from a triage agent to a billing
  or technical subagent. The billing subagent sees only the extracted billing question, not
  the full conversation history that included unrelated troubleshooting.

### Plan-and-execute vs. ReAct

Two control flow strategies for agent systems. **ReAct** (Yao et al., 2022) interleaves
reasoning and action: decide one step, act, observe, repeat. **Plan-and-execute** produces
a full plan first, then works through it, replanning only when something breaks.

| | ReAct | Plan-and-execute |
|---|---|---|
| Planning | None upfront; each step decides the next | Full plan before any execution |
| Adaptability | High; adjusts after every observation | Low until a replan is triggered |
| Predictability | Lower; path depends on intermediate results | Higher; plan is reviewable before execution |
| Failure mode | Wandering without progress | Bad plan stays bad until forced replan |

**Real-world examples:**

- **AutoGPT** (Significant Gravitas, 2023) generates a task list before executing any step,
  then works through the list sequentially. This is plan-and-execute.
- **BabyAGI** (Nakajima, 2023) also plans first but dynamically reprioritizes the task
  queue after each step completes. This is plan-and-execute with online replanning.
- **ChatGPT with tools** (OpenAI) decides one tool call at a time based on the conversation
  so far. This is ReAct.

### Four subagent management patterns

Ordered by how much lifecycle control the orchestrator retains:

1. **Inline tool-call spawn**: calling a subagent looks identical to calling any other tool.
   It blocks until the subagent returns. Simplest pattern.
2. **Fan-out**: multiple independent subagents dispatched in parallel, results collected once
   all finish. Good when subtasks share no dependencies.
3. **Persistent agent pools**: long-lived, stateful workers reused across many tasks rather
   than created fresh each time. Amortizes startup cost for high-throughput systems.
4. **Peer-to-peer teams**: agents message each other directly with no central dispatcher.
   Least centralized, hardest to debug.

**Real-world examples:**

- **CrewAI** (2024) implements the supervisor-worker pattern: one orchestrator assigns roles
  and dispatches tasks to specialized agents.
- **Amazon Bedrock Agents** (AWS, 2024) uses an orchestrator that routes to specialized
  subagents for different domains (database queries, API calls, document retrieval).
- **LangGraph** (LangChain, 2024) supports configurable multi-agent topologies through a
  graph-based state machine where nodes are agents and edges are transitions.

### Cycle detection

Two agents deferring the same decision back and forth, each rephrasing slightly so a
byte-identical duplicate check misses it. Detection requires comparing same-speaker
messages to each other using string similarity, not just checking whether consecutive
messages are identical.

**Real-world example:** Multi-agent code review loops where a reviewer requests changes, the
author makes a minimal edit, the reviewer requests the same change again with different
wording, and the loop continues without progress.

## Section 2: Concept Explanation

### Why subagent isolation matters

An LLM processes its entire context on every call. When a subagent's raw transcript
(tool calls, observations, intermediate reasoning) crosses back to the parent, the parent
pays for those tokens on every subsequent call, and the signal-to-noise ratio drops. A
10-step subagent trace might contain 2,000 tokens of scaffolding wrapping a 50-token answer.
Without compression, the orchestrator carries that 2,000-token payload forward through every
remaining step of its own work.

### Supervisor-worker pipeline (Jack, Bob, Mike)

This chapter uses a consistent three-agent team reused throughout the course:

```
                       +-------+
          task ------->| Jack  |-----> subtask list
                       |planner|
                       +-------+
                           |
                    (for each subtask)
                           |
                           v
                       +-------+
                       | Bob   |-----> assembled draft
                       |worker |
                       +-------+
                     /     |     \
              +------+ +------+ +------+
              |sub-  | |sub-  | |sub-  |  (fresh context each)
              |agent | |agent | |agent |
              +------+ +------+ +------+
                           |
                           v
                       +-------+
                       | Mike  |-----> approved / revise
                       |critic |
                       +-------+
```

Jack breaks the task into subtasks. Bob dispatches one subagent per subtask and assembles
their compressed results into a draft. Mike reviews the draft against required facts. If
Mike says "revise," control loops back to Bob.

### Fan-out vs. sequential dispatch

```
Sequential:               Fan-out:

  subtask A                 subtask A ----> subagent A
      |                     subtask B ----> subagent B   (parallel)
      v                     subtask C ----> subagent C
  subagent A                     |    |    |
      |                          v    v    v
      v                       collect results
  subtask B
      |
      v
  subagent B
      |
      v
  collect results

Total time: sum of all       Total time: max of all
```

Sequential dispatch is simpler but slower. Fan-out reduces wall-clock time when subtasks
are independent, at the cost of higher peak concurrency and more complex error handling
(what happens when 2 of 3 subagents succeed but the third fails?).

### State machines for agent control flow

An agent's control flow can be modeled as a state machine: a fixed set of states (planning,
executing, reviewing, done) with transitions between them. This is what LangGraph makes
concrete: nodes are states, edges are transitions, and some edges are conditional on
what happened in the previous node.

Jack/Bob/Mike as a LangGraph-style state machine:

```
  [jack_plan] ---> [bob_work] ---> [mike_review]
                       ^                |
                       |     "revise"   |
                       +----------------+
                                        |
                              "approved" |
                                        v
                                      [END]
```

### Trade-offs: single agent vs. multi-agent

| | Single agent | Multi-agent team |
|---|---|---|
| Latency | One model call per step | Multiple sequential model calls |
| Cost | One context window | Multiple context windows |
| Debuggability | One trace to read | Multiple interleaved traces |
| Specialization | One prompt covers everything | Each agent's prompt is focused |
| Failure modes | Simpler (loop, hallucination) | More complex (cycles, leaks, redundancy) |

The default should be a single agent. Reach for multi-agent only when you have a concrete
reason: subtasks are independently parallelizable, each requires a specialized prompt or
tool set, or the combined context would exceed the model's window.

## Section 3: Example Code Segments

Tools, brains, and pipeline functions used by the graded tasks and experiments below.

In [ ]:
# Deterministic knowledge base for subagent demos
_TEAM_FACTS = {
    "anthropic founder": (
        "Anthropic was founded in 2021 by Dario Amodei and Daniela Amodei, along with "
        "several colleagues who had previously worked at OpenAI."
    ),
    "react pattern": (
        "ReAct interleaves reasoning traces with actions, letting a model plan and use "
        "tools within the same loop (Yao et al., 2022)."
    ),
}


def mock_search_tool(query: str) -> dict:
    """Look up a fact from the canned knowledge base."""
    q = query.lower()
    for key, fact in _TEAM_FACTS.items():
        # Match on any word in the key appearing in the query
        if key in q or any(w in q for w in key.split()):
            return {"status": "ok", "result": fact}
    return {"status": "not_found", "result": f"No canned result for query: {query!r}"}


# Tool registry the subagent receives
SUBAGENT_TOOLS = {"mock_search": mock_search_tool}

In [ ]:
def subagent_brain(messages: list) -> dict:
    """Rule-based stand-in for the model inside each subagent's ReAct loop."""
    # On first call (no observations yet), search for the subtask
    observations = [m for m in messages if m["role"] == "observation"]
    if not observations:
        return {"action": "mock_search", "action_input": messages[0]["content"]}
    # Once a tool result arrives, return it as the final answer
    return {"action": "final_answer", "action_input": observations[-1]["content"].get("result", "")}


def real_subagent_brain_call(subtask: str) -> str:
    """Real model call for one bounded subtask, fresh isolated context."""
    response = llm_client.call_model(
        messages=[{"role": "user", "content": subtask}],
        system="Answer the user's question directly and concisely in 1-2 sentences.",
        model=llm_client.DEFAULT_MODELS[llm_client.LLM_PROVIDER],
    )
    return response.text

In [ ]:
def real_planner_call(task: str) -> list:
    """Real model call for task decomposition into subtask strings."""
    response = llm_client.call_model(
        messages=[{"role": "user", "content": task}],
        system=(
            "You are a planner. Break the user's task into 2-4 short, self-contained "
            "subtask strings. Respond with ONLY a JSON array of strings, nothing else."
        ),
        model=llm_client.DEFAULT_MODELS[llm_client.LLM_PROVIDER],
    )
    try:
        subtasks = json.loads(response.text)
        if isinstance(subtasks, list) and all(isinstance(s, str) for s in subtasks):
            return subtasks
    except json.JSONDecodeError:
        pass
    # Fallback: treat the whole task as one subtask
    return [task]


def fake_mike(draft: str, required_facts: list) -> dict:
    """Mike, the critic: checks the draft mentions every required fact."""
    missing = [f for f in required_facts if f.lower() not in draft.lower()]
    if missing:
        return {"verdict": "revise", "feedback": f"Missing coverage of: {', '.join(missing)}"}
    return {"verdict": "approved", "feedback": "Covers everything required."}


def real_critic_call(draft: str, required_facts: list) -> dict:
    """Real model call for draft review against required facts."""
    response = llm_client.call_model(
        messages=[{
            "role": "user",
            "content": f"Draft:\n{draft}\n\nRequired facts to cover: {required_facts}",
        }],
        system=(
            "You are a critic reviewing a draft. If it covers all required facts, respond "
            "with exactly: APPROVED. Otherwise respond with: REVISE: <one sentence of "
            "feedback>."
        ),
        model=llm_client.DEFAULT_MODELS[llm_client.LLM_PROVIDER],
    )
    text = response.text.strip()
    if text.upper().startswith("APPROVED"):
        return {"verdict": "approved", "feedback": text}
    return {"verdict": "revise", "feedback": text}

## Section 4: Build It Yourself

Two graded tasks: implement the subagent dispatch loop (context isolation + compressed
return), then the planner that decomposes tasks into subtask lists.

### Task 1: `run_subagent` (subagent dispatch with context isolation)

Implement the inline-tool-call-spawn pattern. The subagent starts from a completely fresh
`messages` list containing only its subtask. It runs its own ReAct loop, and returns only
the final answer (the compressed synthesis) plus the step count.

Both halves of isolation matter independently. *Fresh context in*: the subagent never
sees the parent's messages, so the parent's token cost is not paid twice. *Compressed
result out*: only the answer crosses back, so the parent's context does not fill with the
subagent's scaffolding.

`leaky=True` exists for the Break It section below: it returns the entire raw transcript
instead of the compressed answer.

In [ ]:
def run_subagent(
    subtask: str,
    brain,
    tools: dict,
    max_iterations: int = 4,
    leaky: bool = False,
    verbose: bool = False,
):
    '''Run one subagent on one bounded subtask, in its own fresh context.

    Same ReAct shape as Chapter 1's run_agent(), with three differences:

    1. `messages` starts as [{"role": "user", "content": subtask}] and NOTHING else. Every
       call gets its own new list -- that is the context isolation.
    2. Return a (result, steps_used) tuple rather than a bare string.
    3. With leaky=True, return json.dumps(messages + [the final assistant turn]) instead of
       the compressed answer. That is the bug the break-it section below demonstrates.

    A tool the subagent doesn't have should produce {"status": "error", ...} as its
    observation rather than raising. Running out of iterations returns a message and
    max_iterations.
    '''
    raise NotImplementedError("Implement me, then re-run this cell")


run_subagent = check("ch02-subagent", run_subagent)

In [ ]:
# Demo: run_subagent on a single factual subtask
synthesis, steps_used = run_subagent(
    "who founded anthropic?", subagent_brain, SUBAGENT_TOOLS, verbose=True
)
print(f"\nCompressed synthesis returned to Bob: {synthesis!r}")
print(f"(subagent used {steps_used} step(s) internally)")

### Pipeline setup: planner/critic selection and run_team

Before the next graded task, we set up the full Jack/Bob/Mike pipeline. `run_team`
depends on `run_subagent` (Task 1 above) and the planner function (Task 2 below).

In [ ]:
def run_team(task, required_facts, planner_fn, critic_fn, tracer, verbose=True):
    """Full Jack/Bob/Mike pipeline with tracing."""
    # Jack decomposes the task into subtasks
    tracer.record("Jack (planner)", duration_ms=random.uniform(300, 500), role="planner")
    subtasks = planner_fn(task)
    if verbose:
        print(f"[Jack] Plan: {subtasks}")

    # Bob dispatches one subagent per subtask and assembles results
    notes = []
    for subtask in subtasks:
        tracer.record("Bob -> subagent", duration_ms=random.uniform(250, 450), subtask=subtask)
        if llm_client.HAS_KEY:
            synthesis = real_subagent_brain_call(subtask)
        else:
            synthesis, _ = run_subagent(subtask, subagent_brain, SUBAGENT_TOOLS)
        if verbose:
            print(f"[Bob's subagent] {subtask!r} -> {synthesis!r}")
        notes.append(synthesis)
    draft = " ".join(notes)
    if verbose:
        print(f"[Bob] Draft: {draft}")

    # Mike reviews the assembled draft
    tracer.record("Mike (critic)", duration_ms=random.uniform(200, 350), role="critic")
    review = critic_fn(draft, required_facts)
    if verbose:
        print(f"[Mike] Verdict: {review}")

    return draft, review

### Task 2: `plan_subtasks` (Jack's task decomposition)

Jack's job: split a complex task into self-contained subtask strings that Bob can dispatch
to independent subagents. The interface constraint matters: Bob iterates over whatever Jack
returns, so a plan is always a *list* of strings, even when the task needs no decomposition.
Return a bare string and Bob will iterate over its characters.

In [ ]:
def fake_planner_brain(task: str) -> list:
    '''Jack, the planner -- rule-based stand-in. Recognize this chapter's one demo task
    (it mentions both Anthropic and ReAct) and split it into its two natural subtasks:
    "who founded anthropic?" and "what is the react pattern?".

    Anything else is already atomic: return it as a single-item list.

    Always a list of strings, never a bare string, and never empty -- Bob dispatches one
    subagent per element. Each subtask has to stand on its own, because the subagent that
    receives it starts from a fresh context and will see nothing but that string.
    '''
    raise NotImplementedError("Implement me, then re-run this cell")


fake_planner_brain = check("ch02-planner", fake_planner_brain)

In [ ]:
# Select real or mock brains based on API key availability
planner = real_planner_call if llm_client.HAS_KEY else fake_planner_brain
subagent_call = real_subagent_brain_call if llm_client.HAS_KEY else None
critic = real_critic_call if llm_client.HAS_KEY else fake_mike
print(f"Using {'real model calls' if llm_client.HAS_KEY else 'mock brains'} for Jack, Bob's subagents, and Mike.")

In [ ]:
# Demo: full Jack/Bob/Mike pipeline on the two-part demo task
TASK = "Prepare a short knowledge brief: who founded Anthropic, and what is the ReAct pattern?"
REQUIRED_FACTS = ["anthropic", "react"]

tracer = Tracer()
draft, review = run_team(TASK, REQUIRED_FACTS, planner, critic, tracer)

print("\n=== TRACE ===")
tracer.print_trace()
print(f"\nTotal latency: {tracer.total_ms():.1f}ms")

## Section 5: Playground

Experiments with editable parameters. Change the values and re-run each cell to observe
the effect.

### Experiment 1: Subtask granularity

What happens when Jack splits the task into more or fewer pieces? More subtasks means more
subagent dispatches (more latency, more model calls), but each subagent gets a more focused
question.

In [ ]:
# --- EDIT THESE ---
SUBTASK_COUNTS = [1, 2, 4]  # try [1, 3, 5] or [1, 2, 3, 4, 5]
# ------------------

def planner_with_n_splits(task, n):
    """Force exactly n subtasks by splitting the task text."""
    words = task.split()
    if n <= 1:
        return [task]
    chunk_size = max(1, len(words) // n)
    chunks = []
    for i in range(n):
        start = i * chunk_size
        end = start + chunk_size if i < n - 1 else len(words)
        chunks.append(" ".join(words[start:end]))
    return chunks

for n in SUBTASK_COUNTS:
    t = Tracer()
    subtasks = planner_with_n_splits(TASK, n)
    for st in subtasks:
        t.record(f"subagent: {st[:30]}...", duration_ms=random.uniform(250, 450))
    t.record("Mike (critic)", duration_ms=random.uniform(200, 350))
    print(f"n={n}: {len(subtasks)} subtasks, total latency {t.total_ms():.0f}ms")
    for s in t.spans:
        print(f"  {s.name:40s} {s.duration_ms:>7.1f}ms")
    print()

### Experiment 2: Subagent iteration budget

What happens when you give each subagent more or fewer iterations? A tight budget (2) risks
the subagent running out before finding an answer. A loose budget (10) wastes iterations
if the task is simple.

In [ ]:
# --- EDIT THESE ---
BUDGETS = [2, 5, 10]  # try [1, 3, 6, 20]
# ------------------

for budget in BUDGETS:
    result, steps = run_subagent(
        "who founded anthropic?", subagent_brain, SUBAGENT_TOOLS, max_iterations=budget
    )
    print(f"budget={budget:>2d}: used {steps} step(s), result={result[:60]!r}...")

### Experiment 3: Compression aggressiveness

Compare what Bob receives when the subagent compresses vs. leaks. The word count is a
proxy for context cost (Chapter 5 introduces real tokenization with tiktoken).

In [ ]:
# --- EDIT THESE ---
MAX_WORDS_BUDGET = 15  # try 10, 20, 50 to see truncation thresholds
# ------------------

def count_words(text: str) -> int:
    return len(text.split())

def enforce_compression(raw_result: str, max_words: int = 30) -> str:
    """Hard boundary on subagent result size."""
    words = raw_result.split()
    if len(words) <= max_words:
        return raw_result
    return " ".join(words[:max_words]) + " [...TRUNCATED]"

compressed, _ = run_subagent("who founded anthropic?", subagent_brain, SUBAGENT_TOOLS, leaky=False)
leaked, _ = run_subagent("who founded anthropic?", subagent_brain, SUBAGENT_TOOLS, leaky=True)
truncated = enforce_compression(leaked, max_words=MAX_WORDS_BUDGET)

print(f"Compressed:  {count_words(compressed):>3d} words | {compressed[:80]!r}")
print(f"Leaked:      {count_words(leaked):>3d} words | {leaked[:80]!r}")
print(f"Truncated:   {count_words(truncated):>3d} words | {truncated[:80]!r}")
print(f"\nSize spike without compression: {count_words(leaked)/count_words(compressed):.1f}x")

### Experiment 4: Pipeline length and latency profiling

What happens to total latency as you add agents to the pipeline? Compare 2-agent
(Jack + Bob), 3-agent (Jack + Bob + Mike), and 4-agent (Jack + Bob + Mike + Nina) pipelines.

In [ ]:
# --- EDIT THESE ---
PIPELINE_LENGTHS = [2, 3, 4]  # agents in the pipeline
# ------------------

for n_agents in PIPELINE_LENGTHS:
    t = Tracer()
    t.record("Jack (planner)", duration_ms=random.uniform(300, 500))
    t.record("Bob (worker)", duration_ms=random.uniform(400, 700))
    if n_agents >= 3:
        t.record("Mike (critic)", duration_ms=random.uniform(200, 350))
    if n_agents >= 4:
        t.record("Nina (second reviewer)", duration_ms=random.uniform(200, 350))
    print(f"{n_agents}-agent pipeline: {t.total_ms():.0f}ms total")
    for s in t.spans:
        print(f"  {s.name:30s} {s.duration_ms:>7.1f}ms")
    print()

## Section 6: Break It

Four failure patterns, each demonstrated with the bug first and the fix second. The fourth
scenario includes a graded task.

### Break It 1: Looping subagent with no stop condition

The same infinite-loop failure from Chapter 1, now at the subagent level. A `bait_tool`
always returns "still processing," and the subagent never stops calling it. The fix here
is different from Chapter 1's duplicate-observation detection: a reusable max-iteration
guard that wraps *any* brain function.

In [ ]:
# A tool that never gives a final answer
def bait_tool(_input):
    return {"status": "partial", "detail": "Still processing your request, check back."}

# A brain that keeps calling the bait tool as long as it gets "partial" status
def looping_subagent_brain(messages):
    observations = [m for m in messages if m["role"] == "observation"]
    if not observations or observations[-1]["content"].get("status") == "partial":
        return {"action": "bait_tool", "action_input": "any"}
    return {"action": "final_answer", "action_input": "done"}


print("--- Bug: subagent has no stop condition ---\n")
buggy_result, buggy_iterations = run_subagent(
    "generate the quarterly report subtask",
    looping_subagent_brain,
    tools={"bait_tool": bait_tool},
    max_iterations=15,
    verbose=True,
)
print(f"\nUsed all {buggy_iterations} iterations without finishing.")

In [ ]:
def with_max_iterations(brain_fn, max_iterations: int = 3):
    """Reusable guard: wraps any brain function with an iteration cap."""
    def guarded(messages):
        # Count observations to determine how many actions have been taken
        iterations_so_far = sum(1 for m in messages if m["role"] == "observation")
        if iterations_so_far >= max_iterations:
            return {
                "action": "final_answer",
                "action_input": f"[guard] stopped after {max_iterations} iterations with no progress",
            }
        return brain_fn(messages)
    return guarded


print("--- Fix: same buggy brain, wrapped with a reusable max-iteration guard ---\n")
guarded_brain = with_max_iterations(looping_subagent_brain, max_iterations=3)
fixed_result, fixed_iterations = run_subagent(
    "generate the quarterly report subtask",
    guarded_brain,
    tools={"bait_tool": bait_tool},
    max_iterations=15,
    verbose=True,
)
print(f"\nStopped after {fixed_iterations} iterations (vs {buggy_iterations} before the guard).")

### Break It 2: Jack and Mike stuck in a cycle

A genuine cycle, distinct from a straight-line loop: Mike keeps asking Jack to clarify
scope, and Jack keeps asking Mike what is unclear. Neither message is byte-identical to the
one before it because each rephrases slightly. A naive duplicate-hash check (comparing only
to the immediately preceding message, which is a *different* speaker's text) misses this
entirely. The fix uses string similarity (`difflib`) to compare same-speaker turns.

In [ ]:
# Slightly different phrasings each round to defeat byte-identical checks
JACK_VARIANTS = [
    "Can you clarify what's unclear about the scope, Mike?",
    "Could you clarify what's unclear about the scope, Mike?",
    "Can you please clarify what's unclear about the scope, Mike?",
]
MIKE_VARIANTS = [
    "Ask Jack to clarify the scope before I can review it.",
    "Please ask Jack to clarify the scope before I review it.",
    "Ask Jack to clarify the scope before this can be reviewed.",
]


def cycling_jack(history):
    round_idx = sum(1 for who, _ in history if who == "Jack")
    return JACK_VARIANTS[round_idx % len(JACK_VARIANTS)]


def cycling_mike(history):
    round_idx = sum(1 for who, _ in history if who == "Mike")
    return MIKE_VARIANTS[round_idx % len(MIKE_VARIANTS)]


def detect_cycle(history, lookback=2, threshold=0.6):
    """Compare same-speaker turns using difflib string similarity."""
    by_speaker = {}
    for speaker, msg in history:
        by_speaker.setdefault(speaker, []).append(msg)
    for msgs in by_speaker.values():
        if len(msgs) < lookback:
            continue
        recent = msgs[-lookback:]
        # Check if consecutive same-speaker messages are too similar
        if all(
            difflib.SequenceMatcher(None, recent[i], recent[i + 1]).ratio() >= threshold
            for i in range(len(recent) - 1)
        ):
            return True
    return False


def run_conversation(jack_fn, mike_fn, max_rounds=8, duplicate_guard=False,
                     cycle_guard=False, verbose=True):
    history = []
    last_hash = None
    for round_num in range(1, max_rounds + 1):
        jack_msg = jack_fn(history)
        history.append(("Jack", jack_msg))
        if verbose:
            print(f"[round {round_num}] Jack: {jack_msg}")
        if duplicate_guard:
            h = hash(jack_msg)
            if h == last_hash:
                return f"Stopped by duplicate-hash guard after round {round_num}.", history
            last_hash = h
        if cycle_guard and detect_cycle(history):
            return f"Stopped by semantic cycle guard after round {round_num}.", history

        mike_msg = mike_fn(history)
        history.append(("Mike", mike_msg))
        if verbose:
            print(f"[round {round_num}] Mike: {mike_msg}")
        if duplicate_guard:
            h = hash(mike_msg)
            if h == last_hash:
                return f"Stopped by duplicate-hash guard after round {round_num}.", history
            last_hash = h
        if cycle_guard and detect_cycle(history):
            return f"Stopped by semantic cycle guard after round {round_num}.", history

    return f"Reached max_rounds={max_rounds} without resolution.", history


print("--- Bug: hash-based duplicate check misses the cycle ---\n")
outcome, history = run_conversation(cycling_jack, cycling_mike, duplicate_guard=True)
print("\nOutcome:", outcome)

In [ ]:
print("--- Fix: semantic-similarity cycle detector catches it ---\n")
outcome2, history2 = run_conversation(cycling_jack, cycling_mike, cycle_guard=True)
print("\nOutcome:", outcome2)
print(f"\n(Bug ran the full max_rounds; the fix caught it after {len(history2)} messages.)")

### Break It 3: Unnecessary middle-manager role

Adding a second reviewer, Nina, after Mike. Nina uses identical logic to Mike, so her
verdict always matches. She adds latency with zero new information. The latency profiler
identifies which hop added time with no value.

In [ ]:
def fake_second_reviewer(draft, required_facts):
    # Identical logic to fake_mike: a redundant, duplicate check
    return fake_mike(draft, required_facts)


tracer2 = Tracer()
draft2, mike_review = run_team(TASK, REQUIRED_FACTS, planner, critic, tracer2, verbose=False)
tracer2.record("Nina (second reviewer)", duration_ms=random.uniform(200, 350), role="redundant_reviewer")
nina_review = fake_second_reviewer(draft2, REQUIRED_FACTS)

print("Mike's verdict:", mike_review["verdict"])
print("Nina's verdict:", nina_review["verdict"])
print("Same verdict, zero new information.\n")


def profile_hops(tracer, value_add_hop_names):
    """Label each hop as value-add or not based on a known-good set."""
    return [
        {"name": s.name, "duration_ms": round(s.duration_ms, 1),
         "value_add": s.name in value_add_hop_names}
        for s in tracer.spans
    ]


profile = profile_hops(tracer2, value_add_hop_names={"Jack (planner)", "Bob -> subagent", "Mike (critic)"})
for row in profile:
    flag = "" if row["value_add"] else "  <-- no value-add"
    print(f"{row['name']:30s} {row['duration_ms']:>7.1f}ms{flag}")

no_value_total = sum(r["duration_ms"] for r in profile if not r["value_add"])
print(f"\nLatency with no value-add: {no_value_total:.1f}ms of {tracer2.total_ms():.1f}ms total.")

In [ ]:
def estimate_cost_latency(num_agents, task_complexity, per_hop_latency_ms=350,
                          per_hop_tokens=800, cost_per_million=5.0):
    """Toy cost/latency model for comparing pipeline lengths."""
    multiplier = {"simple": 1, "moderate": 2, "complex": 4}[task_complexity]
    total_latency_ms = num_agents * per_hop_latency_ms * multiplier
    total_tokens = num_agents * per_hop_tokens * multiplier
    total_cost = total_tokens / 1_000_000 * cost_per_million
    return {"latency_ms": total_latency_ms, "tokens": total_tokens, "cost_usd": round(total_cost, 4)}


print(f"{'complexity':10s} {'single-agent':40s} {'3-agent team':40s}")
for complexity in ["simple", "moderate", "complex"]:
    single = estimate_cost_latency(1, complexity)
    multi = estimate_cost_latency(3, complexity)
    print(f"{complexity:10s} {str(single):40s} {str(multi):40s}")

### Break It 4: The leaky subagent

Instead of returning only the final answer, a buggy subagent returns its entire raw
transcript: every thought, every tool call, every observation. The `leaky=True` flag in
`run_subagent` reproduces this bug.

In [ ]:
compressed_result, _ = run_subagent(
    "who founded anthropic?", subagent_brain, SUBAGENT_TOOLS, leaky=False
)
leaky_result, _ = run_subagent(
    "who founded anthropic?", subagent_brain, SUBAGENT_TOOLS, leaky=True
)

print("Compressed synthesis (correct behavior):")
print(" ", compressed_result)
print(f"  words: {count_words(compressed_result)}\n")

print("Leaky full transcript (the bug):")
print(" ", leaky_result)
print(f"  words: {count_words(leaky_result)}\n")

spike = count_words(leaky_result) / count_words(compressed_result)
print(f"Size spike: {spike:.1f}x more content entering Bob's context.")
print("The answer is still in there, but buried inside JSON scaffolding that Bob's")
print("context has to carry forward on every subsequent call.")

#### Detecting a leak by structure, not size

The temptation is a length threshold, since the leaked transcript above was visibly bigger.
That fails: a subagent answering a genuinely broad question returns a long, well-behaved
paragraph, and a subagent that leaks after one step returns a transcript shorter than that
paragraph. What distinguishes a leak is its *shape*: a serialized list of message objects,
each with a `role` and `content` key, rather than prose.

**Hints:**

1. Parse the result as JSON. If parsing fails, it is not a leak.
2. Check whether the parsed value is a non-empty list of dicts, each containing both
   `"role"` and `"content"` keys.

**Production impact:** A leaky subagent inflates the orchestrator's context with every
dispatch. Over a 20-subtask pipeline, a 3x per-subtask size spike compounds to 60x more
tokens in the orchestrator's context than necessary. At \$3 per million input tokens, that
is the difference between \$0.06 and \$3.60 per pipeline run.

**Interview follow-up:** "How do you prevent a subagent from leaking its full
chain-of-thought to the parent?"


In [ ]:
def is_leaky_result(raw_result: str) -> bool:
    '''Did a subagent hand back its raw transcript instead of a compressed synthesis?

    Decide on structure, not size. A leaked transcript is what run_subagent(leaky=True)
    produces: JSON that parses to a non-empty list whose every element is a dict with both
    a "role" and a "content" key.

    Anything else -- prose of any length, JSON scalars, an array of plain strings, an empty
    array, text that merely contains the word "role" -- is not a leak. Return a real bool.
    '''
    raise NotImplementedError("Implement me, then re-run this cell")


is_leaky_result = check("ch02-leak-check", is_leaky_result)

In [ ]:
print("Leak detector on the two results from above:")
print(f"  compressed synthesis -> is_leaky_result = {is_leaky_result(compressed_result)}")
print(f"  leaky transcript     -> is_leaky_result = {is_leaky_result(leaky_result)}")
print()

# The fix: enforce a hard compression boundary on every subagent result
safe_result = enforce_compression(leaky_result)
print("Bob actually receives after enforcement:", safe_result)
print(f"  words: {count_words(safe_result)} (was {count_words(leaky_result)} before)")

## Section 7: Interview Q&A

### Question 1: "When would you use multiple agents vs. a single agent with more tools?"

Start with a single agent. Multiple agents add latency (each hop is another model call),
cost (each maintains its own context window), and failure modes (cycles, leaks, redundant
hops). Reach for multi-agent only when subtasks are independently parallelizable (fan-out
reduces wall-clock time), each requires a specialized prompt or tool set that would bloat
a single agent's context, or the combined context would exceed the model's window. The
Jack/Bob/Mike pipeline in this chapter costs 3x the latency of a single agent doing the
same work sequentially, and that cost is justified only when the task decomposition and
quality review steps produce measurably better output.

### Question 2: "How do you prevent a subagent from leaking sensitive context to the parent?"

Two independent mechanisms. First, structural detection: parse the subagent's return value
and check whether it is a serialized message list (a JSON array of objects with `role` and
`content` keys) rather than prose. A length threshold alone fails because legitimate long
answers and short leaked transcripts overlap in size. Second, a hard compression boundary:
truncate or re-summarize anything above a word budget before it enters the parent's context.
The detector catches the leak; the boundary limits the damage even if detection fails.

### Question 3: "Two agents keep deferring to each other. How do you detect and break the cycle?"

A byte-identical duplicate check misses this because each agent rephrases slightly. The
fix is string similarity on same-speaker turns: compare each speaker's recent messages to
each other (not to the other speaker's messages) using something like `difflib.SequenceMatcher`.
If the last N messages from the same speaker are all above a similarity threshold (0.6 works
in practice), the conversation is stuck. Breaking the cycle requires either escalating to a
human, injecting a concrete decision into the conversation, or killing the loop with an
explicit error. An iteration cap is a complementary backstop that bounds total cost even
if the similarity detector's threshold is set too high.

### Question 4: "What is the cost/latency tradeoff between a single powerful agent and a team of specialized ones?"

Every agent hop adds one model call's worth of latency (300-500ms for small models,
1-2s for large ones) and one context window's worth of token cost. A 3-agent pipeline
costs roughly 3x the latency and 3x the tokens of a single agent on the same task.
The multi-agent approach pays off when specialization produces measurably better output
(e.g., a code-generation agent plus a code-review agent catches bugs a single agent misses),
when fan-out reduces wall-clock time on independent subtasks, or when context isolation
prevents cross-contamination. For simple tasks where a single prompt covers the full
capability, the multi-agent overhead is pure waste.

### Self-check flashcards

Explain multi-agent architecture to a non-technical project manager in 2-3 sentences,
then compare your explanation against the model answer below.

*Model answer:* We split a complex task into smaller pieces and assign each piece to a
specialist. One coordinator decides what the pieces are, workers handle each piece
independently, and a reviewer checks the assembled result before sending it back. This is
slower and more expensive than having one generalist do everything, so we use it only when
the task is complex enough that specialization pays for itself.

### Cold diagnosis exercise

For each symptom below, classify it before checking the model answer. These are phrased
without naming Jack, Bob, or Mike to make this a real diagnosis exercise.

Use `drill.check(n)` after writing your answer to see it alongside the model answer.
`drill.reveal(n)` shows the model answer without requiring your own attempt first.

In [ ]:
from agentlib.self_check import drill as open_drill

drill = open_drill(2)
drill.questions()

In [ ]:
# One slot per question. Replace the placeholder text, then run this cell.
# Answers under 25 words are not recorded -- these need a real attempt.

# Question 1
drill.attempt(1, '''
(Your answer here.)
''')

# Question 2
drill.attempt(2, '''
(Your answer here.)
''')

# Question 3
drill.attempt(3, '''
(Your answer here.)
''')

# Question 4
drill.attempt(4, '''
(Your answer here.)
''')

# Question 5
drill.attempt(5, '''
(Your answer here.)
''')

# Question 6
drill.attempt(6, '''
(Your answer here.)
''')

# Question 7
drill.attempt(7, '''
(Your answer here.)
''')

print()
drill.status()

In [ ]:
# Your answer, then the model answer. Change the number to work through the rest.
drill.check(1)

## Section 8: References

1. Yao, S., Zhao, J., Yu, D., et al. (2022). "ReAct: Synergizing Reasoning and Acting
   in Language Models." arXiv:2210.03629.
2. LangGraph documentation. LangChain, Inc. https://langchain-ai.github.io/langgraph/
3. CrewAI documentation. https://docs.crewai.com/
4. AutoGPT. Significant Gravitas (2023). https://github.com/Significant-Gravitas/AutoGPT
5. BabyAGI. Nakajima, Y. (2023). https://github.com/yoheinakajima/babyagi
6. Anthropic MCP specification (2024). https://modelcontextprotocol.io/

### Related chapters

- **Chapter 1** (Fundamentals): ReAct loop foundation that each subagent runs internally
- **Chapter 5** (Cost, Performance): Measuring the actual cost/latency overhead of
  multi-agent pipelines with real tokenization
- **Chapter 8** (System Design): Using the 9-question framework to decide whether a task
  needs multiple agents at all

## Next: Chapter 3: RAG and Retrieval Evaluation

This chapter's team answered from a tiny, hand-built fact dictionary. Chapter 3 replaces
that with a real retrieval pipeline over real documents, and shows exactly why grounding a
model in retrieved context still does not eliminate hallucination.